In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client()


/home/bbo/Documents/Code/projects/Findcrackeddevs/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/bbo/Documents/Code/projects/Findcrackeddevs/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
# Queries the github_timeline public dataset for the previous month
query = """
    SELECT
        actor.login                            AS login,
        actor.id                               AS actor_id,
        COUNT(*)                               AS n_events,
        COUNT(DISTINCT repo.name)              AS n_repos,
        COUNT(DISTINCT DATE(created_at))       AS active_days,
        COUNTIF(type = 'PushEvent')            AS n_pushes,
        COUNTIF(type = 'PullRequestEvent')     AS n_prs,
        MIN(created_at)                        AS first_seen,
        MAX(created_at)                        AS last_seen,
        ARRAY_AGG(DISTINCT repo.name LIMIT 20) AS repos
    FROM `githubarchive.day.2026*`
    WHERE _TABLE_SUFFIX BETWEEN '1020' AND '1027'
        AND type IN ('PushEvent', 'PullRequestEvent', 'PullRequestReviewEvent')
        AND NOT ENDS_WITH(actor.login, '[bot]')
    GROUP BY login, actor_id
"""
df = client.query(query).to_dataframe()

/home/bbo/Documents/Code/projects/Findcrackeddevs/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [33]:
df.to_parquet('../../data/raw/archive.parquet', index=False)

In [31]:
# Returns a new DataFrame containing only the rows where the value matches
filtered_df = df[df['login'] == 'BilaalBakare']


In [32]:
filtered_df

,login,actor_id,n_events,n_repos,active_days,n_pushes,n_prs,first_seen,last_seen,repos
